# Actividad 2: Simulación en python

In [4]:
# 1. Usar Tkinter para abrir la ventana emergente nativa
%matplotlib qt

import numpy as np
from numpy import sin, cos
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle

# -----------------------------------------------------------
# System parameters
# -----------------------------------------------------------
g = 9.81
l = 1.0        # longitud del péndulo
m = 1.0        # masa del péndulo
M_r = 4.0      # masa del bloque
k = 50.0       # constante elástica del resorte

# -----------------------------------------------------------
# Initial conditions 
# -----------------------------------------------------------
x0 = 0.0                 # Bloque en su posición de equilibrio
v0 = 0.0                 # Bloque parte del reposo
th0 = np.radians(80.0)   # ÁNGULO GRANDE: Péndulo desviado a 80 grados
ome0 = 0.0               # Péndulo parte del reposo

# -----------------------------------------------------------
# Method parameters
# -----------------------------------------------------------
tmax = 20
dt = 0.01
STRIDE = 2

# -----------------------------------------------------------
# Dynamics: pendulum with sliding support on a spring
# -----------------------------------------------------------
def dyn(t, y):
    x, v, th, ome = y
    s, c = sin(th), cos(th)
    den = M_r/m + s**2

    a_x = ((g*c + l*ome**2)*s - (k/m)*x) / den
    a_th = -(1.0/l) * (g*(1 + M_r/m)*s + c*(l*ome**2*s - (k/m)*x)) / den

    return np.array([v, a_x, ome, a_th])

# -----------------------------------------------------------
# Fourth-order Runge-Kutta method
# -----------------------------------------------------------
def rk4(f, t, y, h):
    k1 = h * f(t, y)
    k2 = h * f(t + h/2, y + k1/2)
    k3 = h * f(t + h/2, y + k2/2)
    k4 = h * f(t + h, y + k3)
    return y + (k1 + 2*k2 + 2*k3 + k4) / 6

# -----------------------------------------------------------
# Integration using RK4
# -----------------------------------------------------------
n = int(tmax / dt)
t = np.linspace(0, n*dt, n+1)
y = np.empty((n+1, 4))
y[0] = np.array([x0, v0, th0, ome0])
for i in range(n):
    y[i+1] = rk4(dyn, t[i], y[i], dt)

# -----------------------------------------------------------
# Separate variables after integration
# -----------------------------------------------------------
x = y[:, 0]
v = y[:, 1]
th = y[:, 2]
ome = y[:, 3]

# -----------------------------------------------------------
# Kinematics
# -----------------------------------------------------------
x_block, y_block = x, np.zeros_like(x)
x_bob = x + l*sin(th)
y_bob = -l*cos(th)

# -----------------------------------------------------------
# Figure & Setup
# -----------------------------------------------------------

fig, ax = plt.subplots(figsize=(7, 7))
R = l + max(abs(x).max(), 0.5) + 0.3
ax.set(xlim=(-R, R), ylim=(-R, R/2), aspect='equal', title="Péndulo con soporte deslizante y resorte (RK4)")
ax.grid(alpha=0.3)

# Dimensiones y posiciones fijas
x_wall, wall_w = -R + 0.15, 0.08
block_w, block_h = 0.35, 0.30

# Elementos gráficos
ax.add_patch(Rectangle((x_wall - wall_w, -R/4), wall_w, R/4 + 0.4, fc="white", ec="black", hatch="////", zorder=1))
block = ax.add_patch(Rectangle((x0 - block_w/2, -block_h/2), block_w, block_h, fc="purple", ec="black", zorder=4))
trace, = ax.plot([], [], "-r", lw=1.2, alpha=0.6, zorder=3)
bob, = ax.plot([], [], "o", ms=16, color="purple", mec="black", mew=1.2, zorder=7)
rod, = ax.plot([], [], "-k", lw=1.5, zorder=6)
spring_line, = ax.plot([], [], "-k", lw=1.5, zorder=2)
clock = ax.text(0.05, 0.93, "", transform=ax.transAxes, fontsize=12)

# Vector para la forma relativa del resorte
s_spring = np.linspace(0, 1, 100)
wave_spring = 0.08 * np.sin(24 * np.pi * s_spring)

# -----------------------------------------------------------
# Helper Resorte y Animación
# -----------------------------------------------------------
def spring_coords(x1, x2, n_coils=12):
    """Resorte 1D simplificado a lo largo del eje X."""
    s = np.linspace(0, 1, 150)
    wave = 0.08 * np.sin(2 * np.pi * n_coils * s)
    return x1 + (x2 - x1) * s, wave

def animate(i):
    xb = x_block[i]
    rod.set_data([xb, x_bob[i]], [-block_h/2, y_bob[i]])
    bob.set_data([x_bob[i]], [y_bob[i]])
    block.set_xy((xb - block_w/2, -block_h/2))
    trace.set_data(x_bob[:i+1], y_bob[:i+1])
    
    xs, ys = spring_coords(x_wall, xb - block_w/2)
    spring_line.set_data(xs, ys)
    clock.set_text(f"t = {t[i]:.1f} s")
    return rod, bob, block, trace, spring_line, clock

# -----------------------------------------------------------
# Animation
# -----------------------------------------------------------
ani = FuncAnimation(fig, animate, frames=range(0, n+1, STRIDE),
                     interval=STRIDE*dt*1000, blit=True)

plt.tight_layout()
plt.show()